# Make sure you have ultralytics

In [2]:
!pip install ultralytics

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
  Using cached opencv_python-4.13.0.92-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 970.2 kB/s  0:00:01ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 1.3 MB/s  0:00:05 eta 0:00:01m
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Relabel the football-players-detection to match the Detect Players tags

In [ ]:
import shutil
import sys
from collections import Counter
from pathlib import Path

BASE_DIR = Path("/content")
SOURCE = BASE_DIR / "football-players-detection"
OUTPUT = BASE_DIR / "football-players-detection-relabeled"

# Old class id -> new class id (ball, goalkeeper, player, referee -> Ball, Player, Referee)
CLASS_MAP = {0: 0, 1: 1, 2: 1, 3: 2}
NEW_NAMES = ["Ball", "Player", "Referee"]
SPLITS = ["train", "valid", "test"]


def count_classes(dataset: Path) -> Counter:
    counts = Counter()
    for label_file in dataset.glob("*/labels/*.txt"):
        for line in label_file.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    return counts


def remap_label_file(src: Path, dst: Path) -> None:
    new_lines = []
    for line_number, line in enumerate(src.read_text().splitlines(), start=1):
        if not line.strip():
            continue
        class_id, rest = line.split(maxsplit=1)
        if int(class_id) not in CLASS_MAP:
            sys.exit(f"Unknown class {class_id} in {src} line {line_number}")
        new_lines.append(f"{CLASS_MAP[int(class_id)]} {rest}")
    dst.write_text("\n".join(new_lines) + ("\n" if new_lines else ""))


def write_data_yaml(src: Path, dst: Path) -> None:
    lines = []
    for line in src.read_text().splitlines():
        if line.startswith("nc:"):
            line = f"nc: {len(NEW_NAMES)}"
        elif line.startswith("names:"):
            line = f"names: {NEW_NAMES}"
        lines.append(line)
    dst.write_text("\n".join(lines) + "\n")


if OUTPUT.exists():
    sys.exit(f"{OUTPUT} already exists. Delete it to rebuild the relabeled dataset.")

print("Before:", dict(sorted(count_classes(SOURCE).items())))

for split in SPLITS:
    split_dir = SOURCE / split
    if not split_dir.is_dir():
        continue
    shutil.copytree(split_dir / "images", OUTPUT / split / "images")
    (OUTPUT / split / "labels").mkdir(parents=True)
    for label_file in sorted((split_dir / "labels").glob("*.txt")):
        remap_label_file(label_file, OUTPUT / split / "labels" / label_file.name)

write_data_yaml(SOURCE / "data.yaml", OUTPUT / "data.yaml")
for readme in SOURCE.glob("README*.txt"):
    shutil.copy2(readme, OUTPUT / readme.name)

after = count_classes(OUTPUT)
print("After: ", {NEW_NAMES[class_id]: count for class_id, count in sorted(after.items())})
print(f"Relabeled dataset written to {OUTPUT}")


1181

Train Model (I used google colab but this is how I did it)

In [ ]:
!yolo detect train data=data.yaml model=yolo26s.pt epochs=60 imgsz=1280

Ultralytics 8.4.192 🚀 Python-3.10.2 torch-2.14.1 CPU (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1000, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience